In [0]:
# ─── CÉLULA 1 — CONFIGURAÇÃO E LEITURA DA BRONZE ─────────────────────────────────
# Carrega os dados brutos da camada Bronze para iniciar a limpeza.
# Nomes de tabela devem ser sempre qualificados (catalog.schema.table) para garantir resolução correta no Unity Catalog independente do contexto do cluster.
# F (pyspark.sql.functions) é importado aqui para estar disponível em todas as células seguintes — padrão de importação único por notebook.

from pyspark.sql import functions as F

BRONZE_TABLE = "portfolio.default.telco_bronze"
SILVER_TABLE = "portfolio.default.telco_silver"

df_bronze = spark.read.table(BRONZE_TABLE)

print(f"✔ Linhas lidas da Bronze: {df_bronze.count()}")
df_bronze.printSchema()

✔ Linhas lidas da Bronze: 7043
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: long (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
# ─── CÉLULA 2 — DIAGNÓSTICO DE QUALIDADE DOS DADOS ───────────────────────────────
# Mapeia todos os campos nulos e expõe o problema conhecido do TotalCharges: no CSV original, clientes com tenure=0 têm TotalCharges como string vazia " ", não como NULL, o que impede a conversão direta para double.
# Este diagnóstico orienta as transformações da próxima célula e documenta o estado da camada Bronze para referência futura.

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])
display(null_counts)

problemas = df_bronze.filter(
    (F.col("TotalCharges") == " ") |
    F.col("TotalCharges").isNull()
)
print(f"⚠ Linhas com TotalCharges inválido: {problemas.count()}")
display(problemas.select("customerID", "tenure", "MonthlyCharges", "TotalCharges"))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


⚠ Linhas com TotalCharges inválido: 11


customerID,tenure,MonthlyCharges,TotalCharges
4472-LVYGI,0,52.55,
3115-CZMZD,0,20.25,
5709-LVOEQ,0,80.85,
4367-NUYAO,0,25.75,
1371-DWPAZ,0,56.05,
7644-OMVMY,0,19.85,
3213-VVOLG,0,25.35,
2520-SGTTA,0,20.0,
2923-ARZLG,0,19.7,
4075-WKNIU,0,73.35,


In [0]:
# ─── CÉLULA 3 — CORREÇÃO DE TIPOS E PADRONIZAÇÃO ─────────────────────────────────
# Três classes de correção aplicadas em cadeia (método fluente, sem DataFrames intermediários):

# 1. TotalCharges: string com espaço → 0.0 via WHEN/OTHERWISE antes do cast. Clientes com tenure=0 não geraram cobrança acumulada; 0.0 é semanticamente correto.

# 2. SeniorCitizen: inteiro 0/1 → string "No"/"Yes" para uniformizar com as demais colunas binárias e simplificar o StringIndexer no notebook 04.

# 3. Trim em campos string: remove espaços à direita/esquerda que gerariam categorias duplicadas silenciosamente no StringIndexer (ex: "Yes" ≠ "Yes ").

df_typed = (
    df_bronze
    .withColumn(
        "TotalCharges",
        F.when(
            F.trim(F.col("TotalCharges")) == "",
            F.lit(0.0)
        ).otherwise(F.col("TotalCharges").cast("double"))
    )
    .withColumn(
        "SeniorCitizen",
        F.when(F.col("SeniorCitizen") == 1, "Yes").otherwise("No")
    )
    .withColumn("gender",         F.trim(F.col("gender")))
    .withColumn("PaymentMethod",  F.trim(F.col("PaymentMethod")))
    .withColumn("Contract",       F.trim(F.col("Contract")))
)

print("✔ Tipos após correção:")
df_typed.printSchema()

✔ Tipos após correção:
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: string (nullable = false)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
# ─── CÉLULA 4 — VERIFICAÇÃO VISUAL DA CORREÇÃO FINANCEIRA ────────────────────────
# Exibe amostra das colunas financeiras para confirmar que TotalCharges não contém mais strings vazias e que os valores numéricos estão dentro do range esperado.
# Validação visual antes de comprometer os dados na camada Silver.

display(df_typed.select("customerID", "tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen").limit(10))

customerID,tenure,MonthlyCharges,TotalCharges,SeniorCitizen
7590-VHVEG,1,29.85,29.85,No
5575-GNVDE,34,56.95,1889.5,No
3668-QPYBK,2,53.85,108.15,No
7795-CFOCW,45,42.3,1840.75,No
9237-HQITU,2,70.7,151.65,No
9305-CDSKC,8,99.65,820.5,No
1452-KIOVK,22,89.1,1949.4,No
6713-OKOMC,10,29.75,301.9,No
7892-POOKP,28,104.8,3046.05,No
6388-TABGU,62,56.15,3487.95,No


In [0]:
# ─── CÉLULA 5 — BINARIZAÇÃO DO TARGET E REMOÇÃO DO IDENTIFICADOR ─────────────────
# Churn "Yes"/"No" → 1/0: formato numérico exigido pelo Spark ML para a labelCol nos algoritmos de classificação (LogisticRegression, RandomForest, GBT).
# customerID é removido: identificadores únicos não carregam poder preditivo e, se incluídos, causariam data leakage e overfitting nos modelos.
# A distribuição do target (~26% churn) é exibida para documentar o desbalanceamento de classes — informação relevante para interpretação das métricas no notebook 05.

df_clean = (
    df_typed
    .withColumn(
        "Churn",
        F.when(F.col("Churn") == "Yes", 1).otherwise(0)
    )
    .drop("customerID")
)

display(
    df_clean
    .groupBy("Churn")
    .count()
    .withColumn("pct", F.round(F.col("count") / df_clean.count() * 100, 1))
    .orderBy("Churn")
)

Churn,count,pct
0,5174,73.5
1,1869,26.5


In [0]:
# ─── CÉLULA 6 — ESCRITA NA CAMADA PRATA ──────────────────────────────────────────
# Persiste os dados limpos e tipados na camada Silver do Unity Catalog.
# A Silver armazena dados prontos para análise: sem nulos críticos, tipos corretos e target binarizado — base para o EDA (notebook 03) e Feature Engineering (04).
# saveAsTable com nome qualificado é mandatório no ambiente Serverless.

(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
# ─── CÉLULA 7 — VALIDAÇÃO DA CAMADA PRATA ────────────────────────────────────────
# Suite de quatro validações que garantem a integridade antes de avançar para o EDA:
# 1. Contagem igual à Bronze: confirma que nenhuma linha foi descartada.
# 2. Zero nulos em TotalCharges: verifica que o tratamento de strings vazias funcionou.
# 3. Tipo double em TotalCharges: garante compatibilidade com operações numéricas.
# 4. Apenas {0, 1} em Churn: confirma binarização correta do target.

df_silver = spark.read.table(SILVER_TABLE)

print(f"✔ Linhas Bronze : {df_bronze.count()}")
print(f"✔ Linhas Prata  : {df_silver.count()}")

nulls_restantes = df_silver.filter(F.col("TotalCharges").isNull()).count()
print(f"✔ Nulos em TotalCharges: {nulls_restantes}  (esperado: 0)")

print(f"✔ Tipo de TotalCharges: {dict(df_silver.dtypes)['TotalCharges']}  (esperado: double)")

valores_unicos = [row[0] for row in df_silver.select("Churn").distinct().collect()]
print(f"✔ Valores únicos em Churn: {sorted(valores_unicos)}")

✔ Linhas Bronze : 7043
✔ Linhas Prata  : 7043
✔ Nulos em TotalCharges: 0  (esperado: 0)
✔ Tipo de TotalCharges: double  (esperado: double)
✔ Valores únicos em Churn: [0, 1]
